In [1]:
import pandas as pd
import numpy as np

# Archivos
pred = pd.read_csv("predictions_with_probs.csv")
mapping = pd.read_csv("analytical_sample_item_counts_mapping.csv")

# idx_original en predictions_with_probs.csv corresponde al índice filtrado/analítico
idx = pred["idx_original"].astype(int).values
y_true_model = pred["y_true"].astype(int).values

# Definiciones candidatas
y_count_based = mapping.iloc[idx]["overlap_ge1_count_based"].astype(int).values
y_target_flag = mapping.iloc[idx]["target_overlap_flag"].astype(int).values

print("Count-based matches:", (y_true_model == y_count_based).sum(), "/", len(y_true_model))
print("Target flag matches:", (y_true_model == y_target_flag).sum(), "/", len(y_true_model))

print("Model test positives:", y_true_model.sum())
print("Count-based test positives:", y_count_based.sum())
print("Target flag test positives:", y_target_flag.sum())

# Casos discrepantes entre target flag y y_true del modelo
mismatch = idx[y_true_model != y_target_flag]
print("Mismatching filtered indices:", mismatch)
print(mapping.iloc[mismatch][[
    "filtered_idx",
    "original_idx",
    "victim_count",
    "perp_count",
    "overlap_ge1_count_based",
    "target_overlap_flag"
]])

Count-based matches: 942 / 942
Target flag matches: 940 / 942
Model test positives: 178
Count-based test positives: 178
Target flag test positives: 180
Mismatching filtered indices: [2341 2433]
      filtered_idx  original_idx  victim_count  perp_count  \
2341          2341          2494             2           0   
2433          2433          2592             0           0   

      overlap_ge1_count_based  target_overlap_flag  
2341                        0                    1  
2433                        0                    1  


Perfecto: eso confirma al 100% que el modelo final de overlap usó el outcome count-based:

overlap = V.SUM.TOTAL >= 1 AND P.SUM.TOTAL >= 1

No usó exactamente el flag VICTIMA_PERPETRADOR.

La prueba clave es esta:

Count-based matches: 942 / 942
Target flag matches: 940 / 942
Model test positives: 178
Count-based test positives: 178
Target flag test positives: 180

Es decir:

Definición	Coincide con el y_true del modelo
V.SUM.TOTAL >= 1 AND P.SUM.TOTAL >= 1	942/942
VICTIMA_PERPETRADOR	940/942

Y los dos casos discrepantes son claros:

filtered_idx 2341 / original_idx 2494: victim_count = 2, perp_count = 0, flag overlap = 1
filtered_idx 2433 / original_idx 2592: victim_count = 0, perp_count = 0, flag overlap = 1

Estos dos no pueden ser overlap según la definición metodológica del manuscrito, porque no tienen perpetración positiva; el segundo ni victimización ni perpetración. Por tanto, parecen errores o residuos de una variable flag antigua.


In [1]:
import pandas as pd

mapping = pd.read_csv("analytical_sample_item_counts_mapping.csv")

print(mapping.shape)
print(mapping.columns.tolist())

print("Count-based positives:", mapping["overlap_ge1_count_based"].sum())
print("Target flag positives:", mapping["target_overlap_flag"].sum())

mismatch_all = mapping[
    mapping["overlap_ge1_count_based"].astype(int) != mapping["target_overlap_flag"].astype(int)
]

print("Number of mismatches:", len(mismatch_all))

display(mismatch_all[[
    "filtered_idx",
    "original_idx",
    "victim_count",
    "perp_count",
    "overlap_ge1_count_based",
    "target_overlap_flag"
]])

(3767, 14)
['filtered_idx', 'original_idx', 'victim_count', 'perp_count', 'victim_ge1', 'victim_ge2', 'victim_ge3', 'perp_ge1', 'perp_ge2', 'perp_ge3', 'overlap_ge1_count_based', 'overlap_ge2', 'overlap_ge3', 'target_overlap_flag']
Count-based positives: 713
Target flag positives: 719
Number of mismatches: 6


,filtered_idx,original_idx,victim_count,perp_count,overlap_ge1_count_based,target_overlap_flag
2341,2341,2494,2,0,0,1
2342,2342,2496,4,0,0,1
2348,2348,2503,4,0,0,1
2425,2425,2584,0,0,0,1
2433,2433,2592,0,0,0,1
2434,2434,2593,0,0,0,1
